# Ingredient Sanity Check

***

### Imports

In [1]:
import pandas as pd

In [2]:
ingredients = pd.read_csv("data/preprocessed/AMS_data/Ingredients_List.csv")
items = pd.read_csv("data/preprocessed/AMS_data/Items_List.csv")
preps = pd.read_csv("data/preprocessed/AMS_data/Preps_List.csv")
products = pd.read_csv("data/preprocessed/AMS_data/Products_List.csv")
mapping = pd.read_csv("data/mapping/AMS_data/Mapping.csv")
RESTAURANT_NAME = "Gallery_25"

In [ ]:
def get_ingredients(recipe_id):
    """
    Retrieve all ingredients and their quantities for a given recipe.
    This function takes a recipe ID and returns a DataFrame containing all the ingredients
    required for the recipe, along with their quantities. It handles nested ingredients
    by recursively fetching all related ingredients.
    Parameters:
    recipe_id (str): The ID of the recipe for which ingredients are to be retrieved.
    Returns:
    pd.DataFrame: A DataFrame containing the ingredients with their quantities.
    """
    recipe = ingredients[ingredients["Recipe"] == recipe_id]
    def get_all_ingredients(recipe):
        all_ingredients = pd.DataFrame()  # Initialize an empty DataFrame to store all ingredients

        for index, row in recipe.iterrows():
            temp = ingredients[ingredients["Recipe"] == row["IngredientId"]]
            if not temp.empty:
                all_ingredients = pd.concat([all_ingredients, temp, get_all_ingredients(temp)], ignore_index=True)
            all_ingredients = pd.concat([all_ingredients, temp], ignore_index=True)
        
        return pd.concat([recipe, all_ingredients], ignore_index=True)  # Return original recipe with all ingredients

    # Calling the function with the recipe DataFrame
    complete_recipe = get_all_ingredients(recipe)
    recipe_items = []
    recipe_preps = []

    # Seperate into Items and Preps
    items_list = complete_recipe[complete_recipe["IngredientId"].str.startswith("I")]
    items_list.drop_duplicates(subset="IngredientId", keep="first", inplace=True)
    preps_list = complete_recipe[complete_recipe["IngredientId"].str.startswith("P")]
    preps_list.drop_duplicates(subset="IngredientId", keep="first", inplace=True)

    for index,row in complete_recipe.iterrows():
        temp = items[items["ItemId"] == row["IngredientId"]]
        recipe_items.append(temp)
        temp2 = preps[preps["PrepId"] == row["IngredientId"]]
        recipe_preps.append(temp2)
    
    recipe_items = pd.concat(recipe_items, ignore_index=True)
    recipe_preps = pd.concat(recipe_preps, ignore_index=True)
    recipe_items.drop_duplicates(subset="ItemId", keep="first", inplace=True)
    recipe_preps.drop_duplicates(subset="PrepId", keep="first", inplace=True)
    assert len(recipe_items) == len(items_list)
    recipe_items_with_quants = pd.merge(recipe_items, items_list, left_on="ItemId", right_on="IngredientId")
    recipe_items_with_quants.drop(columns=["IngredientId","CaseQty","CaseUOM","PakQty","PakUOM", "InventoryGroup"], inplace=True)
    return recipe_items_with_quants

In [4]:
# Initialize a list to store all recipe and ingredient details
data = []

for _, row in products.iterrows():
    # Extract recipe details
    recipe_id = row['ProdId']
    recipe_name = row.get('Description', 'Unknown')
    
    # Get the ingredients for the recipe
    ingres = get_ingredients(row["ProdId"])
    
    for _, ingredient in ingres.iterrows():
        # Filter the mapping for the current ingredient
        item_mapping = mapping[mapping["ItemId"] == ingredient["ItemId"]]
        
        # Safely extract values from item_mapping
        category_id = item_mapping['CategoryID'].iloc[0] if not item_mapping.empty else "Unknown"
        food_category = item_mapping['Food Category_y'].iloc[0] if not item_mapping.empty else "Unknown"
        
        # Append a dictionary with all details to the data list
        data.append({
            "Recipe ID": recipe_id,
            "Recipe Name": recipe_name,
            "Ingredient": ingredient['Description'],
            "Quantity": ingredient['Qty'],
            "Unit": ingredient['Uom'],
            "Category ID": category_id,
            "Emission Category": food_category,
        })
# List of preps to be included
preps_list = []

# # Uncommennt the below code if all preps are to be included
# for index, row in preps.iterrows():
#     preps_list.append(preps.loc[index, 'PrepId'])

for prep in preps_list:
    recipe_id = prep
    try:
        recipe_name = preps[preps["PrepId"] == prep]["Description"].iloc[0]
    except:
        continue

    display(preps[preps["PrepId"] == prep])
    
    # Get the ingredients for the recipe
    ingres = get_ingredients(recipe_id)
    
    for _, ingredient in ingres.iterrows():
        # Filter the mapping for the current ingredient
        item_mapping = mapping[mapping["ItemId"] == ingredient["ItemId"]]
        
        # Safely extract values from item_mapping
        category_id = item_mapping['CategoryID'].iloc[0] if not item_mapping.empty else "Unknown"
        food_category = item_mapping['Food Category_y'].iloc[0] if not item_mapping.empty else "Unknown"
        
        # Append a dictionary with all details to the data list
        data.append({
            "Recipe ID": recipe_id,
            "Recipe Name": recipe_name,
            "Ingredient": ingredient['Description'],
            "Quantity": ingredient['Qty'],
            "Unit": ingredient['Uom'],
            "Category ID": category_id,
            "Emission Category": food_category,
        })


    IngredientId   Qty Uom  Recipe
35        I-1833   1.0  ea  P-2535
36        I-4133   1.0  ea  P-2535
37       I-14530  30.0   g  P-2535
38        I-5204  20.0   g  P-2535
39        P-2519  10.0  ml  P-2535
292       I-1833   1.0  ea  P-2535
293       I-4133   1.0  ea  P-2535
294      I-14530  30.0   g  P-2535
295       I-5204  20.0   g  P-2535
296       P-2519  10.0  ml  P-2535
Empty DataFrame
Columns: [IngredientId, Qty, Uom, Recipe]
Index: []
Empty DataFrame
Columns: [IngredientId, Qty, Uom, Recipe]
Index: []
Empty DataFrame
Columns: [IngredientId, Qty, Uom, Recipe]
Index: []
Empty DataFrame
Columns: [IngredientId, Qty, Uom, Recipe]
Index: []
    IngredientId    Qty Uom  Recipe
156       I-1941  100.0  ml  P-2519
157       I-2436   50.0  ml  P-2519
158       I-2119  500.0  ml  P-2519
159      I-14790  200.0  ml  P-2519
160       I-7980   50.0  ml  P-2519
161       I-2367   40.0   g  P-2519
162       I-3258   20.0   g  P-2519
163       I-1874   10.0   g  P-2519
164       I-4091   

/var/folders/tw/grzx75dd27n13qjnqlm37zk80000gn/T/ipykernel_14223/1740044415.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  items_list.drop_duplicates(subset="IngredientId", keep="first", inplace=True)
/var/folders/tw/grzx75dd27n13qjnqlm37zk80000gn/T/ipykernel_14223/1740044415.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  preps_list.drop_duplicates(subset="IngredientId", keep="first", inplace=True)
/var/folders/tw/grzx75dd27n13qjnqlm37zk80000gn/T/ipykernel_14223/1740044415.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation:

    IngredientId    Qty      Uom  Recipe
53        I-4136   1.00       ea  P-9279
54        I-4144   6.00       ea  P-9279
55        I-1789   0.03     HEAD  P-9279
56        I-4138   2.00       ml  P-9279
57       P-16862   1.00  oz (fl)  P-9279
58        P-7924   0.70       ea  P-9279
59        I-4140  50.00        g  P-9279
60       P-16221   5.00        g  P-9279
61        I-4107   0.02    bunch  P-9279
62        I-4143   5.00        g  P-9279
297       I-4136   1.00       ea  P-9279
298       I-4144   6.00       ea  P-9279
299       I-1789   0.03     HEAD  P-9279
300       I-4138   2.00       ml  P-9279
301      P-16862   1.00  oz (fl)  P-9279
302       P-7924   0.70       ea  P-9279
303       I-4140  50.00        g  P-9279
304      P-16221   5.00        g  P-9279
305       I-4107   0.02    bunch  P-9279
306       I-4143   5.00        g  P-9279
Empty DataFrame
Columns: [IngredientId, Qty, Uom, Recipe]
Index: []
Empty DataFrame
Columns: [IngredientId, Qty, Uom, Recipe]
Index: []
Emp

/var/folders/tw/grzx75dd27n13qjnqlm37zk80000gn/T/ipykernel_14223/1740044415.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  items_list.drop_duplicates(subset="IngredientId", keep="first", inplace=True)
/var/folders/tw/grzx75dd27n13qjnqlm37zk80000gn/T/ipykernel_14223/1740044415.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  preps_list.drop_duplicates(subset="IngredientId", keep="first", inplace=True)
/var/folders/tw/grzx75dd27n13qjnqlm37zk80000gn/T/ipykernel_14223/1740044415.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation:

    IngredientId     Qty    Uom  Recipe
93        P-4733   50.00      g  P-4611
94        I-4140  150.00      g  P-4611
95       P-11125   20.00      g  P-4611
96        P-3045    1.00     ea  P-4611
97       P-16221    5.00      g  P-4611
98       P-16222    5.00      g  P-4611
99        I-4143    1.00      g  P-4611
100       I-4107    0.01  bunch  P-4611
323       P-4733   50.00      g  P-4611
324       I-4140  150.00      g  P-4611
325      P-11125   20.00      g  P-4611
326       P-3045    1.00     ea  P-4611
327      P-16221    5.00      g  P-4611
328      P-16222    5.00      g  P-4611
329       I-4143    1.00      g  P-4611
330       I-4107    0.01  bunch  P-4611
    IngredientId   Qty    Uom  Recipe
247       I-1927  15.0      g  P-4733
248       I-4126   7.0      g  P-4733
249       I-4124   7.0      g  P-4733
250      I-15554   7.0      g  P-4733
251       I-4107   0.2  bunch  P-4733
252      I-14429  25.0      g  P-4733
253       I-4091   1.0      g  P-4733
Empty DataFrame


/var/folders/tw/grzx75dd27n13qjnqlm37zk80000gn/T/ipykernel_14223/1740044415.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  items_list.drop_duplicates(subset="IngredientId", keep="first", inplace=True)
/var/folders/tw/grzx75dd27n13qjnqlm37zk80000gn/T/ipykernel_14223/1740044415.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  preps_list.drop_duplicates(subset="IngredientId", keep="first", inplace=True)
/var/folders/tw/grzx75dd27n13qjnqlm37zk80000gn/T/ipykernel_14223/1740044415.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation:

    IngredientId   Qty Uom  Recipe
229       I-2367  16.0   g  P-2139
230       I-1874  12.0   g  P-2139
231      I-15547  40.0   g  P-2139
232       I-2263  40.0   g  P-2139
233      I-14422   8.0   g  P-2139
234      I-12764  22.0  ml  P-2139
235       I-4110  26.0   g  P-2139
423       I-2367  16.0   g  P-2139
424       I-1874  12.0   g  P-2139
425      I-15547  40.0   g  P-2139
426       I-2263  40.0   g  P-2139
427      I-14422   8.0   g  P-2139
428      I-12764  22.0  ml  P-2139
429       I-4110  26.0   g  P-2139
Empty DataFrame
Columns: [IngredientId, Qty, Uom, Recipe]
Index: []
Empty DataFrame
Columns: [IngredientId, Qty, Uom, Recipe]
Index: []
Empty DataFrame
Columns: [IngredientId, Qty, Uom, Recipe]
Index: []
Empty DataFrame
Columns: [IngredientId, Qty, Uom, Recipe]
Index: []
Empty DataFrame
Columns: [IngredientId, Qty, Uom, Recipe]
Index: []
Empty DataFrame
Columns: [IngredientId, Qty, Uom, Recipe]
Index: []
Empty DataFrame
Columns: [IngredientId, Qty, Uom, Recipe]
Index: []

/var/folders/tw/grzx75dd27n13qjnqlm37zk80000gn/T/ipykernel_14223/1740044415.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  items_list.drop_duplicates(subset="IngredientId", keep="first", inplace=True)
/var/folders/tw/grzx75dd27n13qjnqlm37zk80000gn/T/ipykernel_14223/1740044415.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  preps_list.drop_duplicates(subset="IngredientId", keep="first", inplace=True)
/var/folders/tw/grzx75dd27n13qjnqlm37zk80000gn/T/ipykernel_14223/1740044415.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation:

    IngredientId    Qty Uom  Recipe
8         I-2119   60.0  ml  P-2679
9         I-4118  450.0   g  P-2679
278       I-2119   60.0  ml  P-2679
279       I-4118  450.0   g  P-2679
Empty DataFrame
Columns: [IngredientId, Qty, Uom, Recipe]
Index: []
Empty DataFrame
Columns: [IngredientId, Qty, Uom, Recipe]
Index: []
Empty DataFrame
Columns: [IngredientId, Qty, Uom, Recipe]
Index: []
Empty DataFrame
Columns: [IngredientId, Qty, Uom, Recipe]
Index: []
    IngredientId  Qty Uom  Recipe
124       I-1982  4.0   L  P-6574
125      P-16855  1.0   L  P-6574
351       I-1982  4.0   L  P-6574
352      P-16855  1.0   L  P-6574
Empty DataFrame
Columns: [IngredientId, Qty, Uom, Recipe]
Index: []
  IngredientId    Qty Uom   Recipe
6       I-1874  120.0   g  P-16855
7       I-2119  300.0  ml  P-16855
Empty DataFrame
Columns: [IngredientId, Qty, Uom, Recipe]
Index: []
Empty DataFrame
Columns: [IngredientId, Qty, Uom, Recipe]
Index: []
Empty DataFrame
Columns: [IngredientId, Qty, Uom, Recipe]
Index: []
 

/var/folders/tw/grzx75dd27n13qjnqlm37zk80000gn/T/ipykernel_14223/1740044415.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  items_list.drop_duplicates(subset="IngredientId", keep="first", inplace=True)
/var/folders/tw/grzx75dd27n13qjnqlm37zk80000gn/T/ipykernel_14223/1740044415.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  preps_list.drop_duplicates(subset="IngredientId", keep="first", inplace=True)
/var/folders/tw/grzx75dd27n13qjnqlm37zk80000gn/T/ipykernel_14223/1740044415.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation:

In [5]:
# Create a DataFrame from the collected data
df = pd.DataFrame(data)

Still have to consider that some items will only have 1 item in the recipe because sometimes the item is the product itself

In [6]:
## MANUAL CHECK
# A way to check for items that are not made in house
df[df["Recipe ID"].map(df["Recipe ID"].value_counts()) == 1]

,Recipe ID,Recipe Name,Ingredient,Quantity,Unit,Category ID,Emission Category


In [7]:
# Save the DataFrame to a CSV file
df.to_csv(f"ingredients_{RESTAURANT_NAME}.csv", index=False)
df.to_excel(f"ingredients_{RESTAURANT_NAME}.xlsx", sheet_name="Labels", index=False)
print("Data successfully saved to ingredients_"+RESTAURANT_NAME+".csv")

Data successfully saved to ingredients_Gallery_25.csv
